# 11 — New Encoders (v0.6.0)

Four new encoders introduced in v0.6.0:

| Encoder | Strategy | Qubits | Depth |
|---|---|---|---|
| `ZZFeatureMapEncoder` | H + Rz + ZZ interactions (Qiskit convention) | d | O(d² · reps) |
| `PauliFeatureMapEncoder` | Configurable Pauli string interactions | d | O(d² · reps) |
| `RandomFourierEncoder` | RBF kernel approximation (requires fit) | n_components | O(1) |
| `TensorProductEncoder` | Ry + Rz per qubit, 2 features/qubit | ⌈d/2⌉ | O(1) |

In [ ]:
import numpy as np

import quprep as qd
from quprep.encode.pauli_feature_map import PauliFeatureMapEncoder
from quprep.encode.random_fourier import RandomFourierEncoder
from quprep.encode.tensor_product import TensorProductEncoder
from quprep.encode.zz_feature_map import ZZFeatureMapEncoder

rng = np.random.default_rng(42)
exporter = qd.QASMExporter()

x3 = np.array([0.5, 1.2, 0.75]) * np.pi
x4 = np.array([0.3, 0.9, 0.6, 1.1]) * np.pi

## 1. ZZ Feature Map

Havlíček et al. 2019 (Qiskit convention). Applies H + single-qubit Rz + pairwise ZZ interactions, repeated `reps` times.

$$\phi_i = 2(\pi - x_i), \quad \phi_{ij} = 2(\pi - x_i)(\pi - x_j)$$

In [ ]:
enc_zz = ZZFeatureMapEncoder(reps=2)
result_zz = enc_zz.encode(x3)

print(f"n_qubits      : {result_zz.metadata['n_qubits']}")
print(f"reps          : {result_zz.metadata['reps']}")
print(f"pairs         : {result_zz.metadata['pairs']}")
print(f"single_angles : {[round(a, 4) for a in result_zz.metadata['single_angles']]}")
print(f"pair_angles   : {[round(a, 4) for a in result_zz.metadata['pair_angles']]}")
print()
print(exporter.export(result_zz))

## 2. Pauli Feature Map

Generalized feature map with configurable Pauli string interactions.

- **Single Paulis:** `X`, `Y`, `Z`
- **Pair Paulis:** `XX`, `YY`, `ZZ`, `XZ`, `ZX`, `XY`, `YX`, `YZ`, `ZY`

In [ ]:
# Z + ZZ — equivalent to ZZFeatureMap
enc_p = PauliFeatureMapEncoder(paulis=["Z", "ZZ"], reps=1)
result_p = enc_p.encode(x3)

print("paulis=[Z, ZZ], reps=1")
st = {k: [round(v2, 4) for v2 in v] for k, v in result_p.metadata["single_terms"].items()}
pt = {k: [(i, j, round(a, 4)) for i, j, a in v] for k, v in result_p.metadata["pair_terms"].items()}
print(f"  single_terms : {st}")
print(f"  pair_terms   : {pt}")
print()
print(exporter.export(result_p))

In [ ]:
# Mixed Paulis — higher expressivity
enc_p2 = PauliFeatureMapEncoder(paulis=["Z", "X", "ZZ"], reps=1)
result_p2 = enc_p2.encode(x3)

print("paulis=[Z, X, ZZ], reps=1")
print(f"  single_terms : { {k: len(v) for k, v in result_p2.metadata['single_terms'].items()} }")
print(f"  pair_terms   : { {k: len(v) for k, v in result_p2.metadata['pair_terms'].items()} }")
print(f"  depth        : {result_p2.metadata['depth']}")

## 3. Random Fourier Features

Approximates the RBF (Gaussian) kernel via Bochner's theorem.

$$\phi(x) = \sqrt{\frac{2}{D}} \cos(Wx + b), \quad W \sim \mathcal{N}(0, 2\gamma), \quad b \sim \mathcal{U}(0, 2\pi)$$

> **Note:** Must call `fit()` before `encode()`. The number of output qubits is always `n_components`, regardless of input dimension.

In [ ]:
X_train = rng.uniform(-1, 1, size=(100, 4))  # 100 samples, 4 features
x_new   = rng.uniform(-1, 1, 4)

enc_rf = RandomFourierEncoder(n_components=8, gamma=1.0, random_state=42)
enc_rf.fit(X_train)  # samples W and b

result_rf = enc_rf.encode(x_new)

print(f"Input dim     : {len(x_new)}  (4 features)")
print(f"n_qubits      : {result_rf.metadata['n_qubits']}  (always n_components)")
print(f"output range  : [{result_rf.parameters.min():.4f}, {result_rf.parameters.max():.4f}]")
print()
print(exporter.export(result_rf))

## 4. Tensor Product Encoding

Encodes two features per qubit using a full Bloch sphere rotation (Ry + Rz). Requires $\lceil d/2 \rceil$ qubits. No entanglement.

$$|\psi_k\rangle = R_z(\theta_{2k+1}) R_y(\theta_{2k}) |0\rangle$$

In [ ]:
enc_tp = TensorProductEncoder()
result_tp = enc_tp.encode(x4)

print(f"Input dim  : {len(x4)}")
print(f"n_qubits   : {result_tp.metadata['n_qubits']}  (ceil(4/2) = 2)")
print(f"ry_angles  : {[round(a, 4) for a in result_tp.metadata['ry_angles']]}")
print(f"rz_angles  : {[round(a, 4) for a in result_tp.metadata['rz_angles']]}")
print()
print(exporter.export(result_tp))

In [ ]:
# Odd-length input — last Rz is zero-padded
x5 = np.array([0.3, 0.8, 1.2, 0.5, 0.9]) * np.pi
result_tp5 = enc_tp.encode(x5)
print(f"5 features → n_qubits: {result_tp5.metadata['n_qubits']}  (ceil(5/2) = 3, last rz = 0)")

## 5. Via `prepare()`

All stateless encoders work directly with the `prepare()` one-liner.

In [ ]:
X = rng.uniform(0, 1, size=(10, 4))

for enc_name in ("zz_feature_map", "tensor_product"):
    result = qd.prepare(X, encoding=enc_name)
    meta = result.encoded[0].metadata
    print(f"  {enc_name:20s}  qubits={meta['n_qubits']}  depth={meta['depth']}")

# RandomFourier requires fit — use the encoder directly
enc_rf2 = RandomFourierEncoder(n_components=6, random_state=0)
enc_rf2.fit(X)
encoded_list = [enc_rf2.encode(X[i]) for i in range(len(X))]
print(f"  {'random_fourier':20s}  qubits={encoded_list[0].metadata['n_qubits']}  (6 components)")